In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start), COUNT(*)
FROM prod.detection.viewing_commercials_golden
WHERE session_start >= '2025-06-17 14:00:00'
AND MOD(fk_tvid, 10) = 1
GROUP BY 1
ORDER BY 1 DESC

In [0]:
%sql
SELECT DATE_TRUNC('HOUR', session_start), COUNT(*)
FROM prod.detection.viewing_commercials_firehose
WHERE session_start >= '2025-06-17 14:00:00'
AND MOD(fk_tvid, 10) = 1
GROUP BY 1
ORDER BY 1 DESC

In [0]:
def get_column_names(vendor_name):
    columns = "tvid string, hash string, zipcode string, dma string, value string, mt_start integer, ts_start timestamp, ts_end timestamp, "
    if vendor == 'TMS':
        columns += 'prev_episode_id_tms string, prev_title_tms string, prev_ts_start timestamp, prev_ts_end timestamp, prev_channel_callsign_tms string, prev_network_affiliate_tms string, next_episode_id_tms string, next_title_tms string, next_ts_start timestamp, next_ts_end timestamp, next_channel_callsign_tms string, next_network_affiliate_tms string, live string, brand_name string, title string, duration integer, ip string, input_category string, input_device string, app_service_tms string'
    else:
        columns += 'prev_episode_id string, prev_title string, prev_ts_start timestamp, prev_ts_end timestamp, prev_channel_callsign string, prev_network_affiliate string, next_episode_id string, next_title string, next_ts_start timestamp, next_ts_end timestamp, next_channel_callsign string, next_network_affiliate string, live string, brand_name string, title string, duration integer, ip string, input_category string, input_device string, app_service string'
    return columns

In [0]:
def column_names_insert(vendor_name):
    columns = 'tvid, hash, zipcode, dma, value, mt_start, ts_start, ts_end, '
    if vendor_name == 'TMS':
        columns += 'prev_episode_id_tms, prev_title_tms, prev_ts_start, prev_ts_end, prev_channel_callsign_tms, prev_network_affiliate_tms, next_episode_id_tms, next_title_tms, next_ts_start, next_ts_end, next_channel_callsign_tms, next_network_affiliate_tms, live, brand_name, title, duration, ip, input_category, input_device, app_service_tms'
    else:
        columns += 'prev_episode_id, prev_title, prev_ts_start, prev_ts_end, prev_channel_callsign, prev_network_affiliate, next_episode_id, next_title, next_ts_start, next_ts_end, next_channel_callsign, next_network_affiliate, live, brand_name, title, duration, ip, input_category, input_device, app_service'
    return columns

In [0]:
def prev_next_cols(vendor, prev_or_next, col_type):
    if col_type == 'station':
        if prev_or_next == 'prev':
            prev_station_id = 'c.prev_station_id' if vendor == 'TIVO' else 'c.tms_prev_station_id'
            other_prev_station_id = 'c.tms_prev_station_id' if vendor == 'TIVO' else 'c.prev_station_id'
            return prev_station_id, other_prev_station_id
        elif prev_or_next == 'next':
            next_station_id = 'c.next_station_id' if vendor == 'TIVO' else 'c.tms_next_station_id'
            other_next_station_id = 'c.tms_next_station_id' if vendor == 'TIVO' else 'c.next_station_id'
            return next_station_id, other_next_station_id
    elif col_type == 'show':
        if prev_or_next == 'prev':
            prev_show_id = 'c.prev_show_id' if vendor == 'TIVO' else 'c.tms_prev_show_id'
            other_prev_show_id = 'c.tms_prev_show_id' if vendor == 'TIVO' else 'c.prev_show_id'
            return prev_show_id, other_prev_show_id
        elif prev_or_next == 'next':
            next_show_id = 'c.next_show_id' if vendor == 'TIVO' else 'c.tms_next_show_id'
            other_next_show_id = 'c.tms_next_show_id' if vendor == 'TIVO' else 'c.next_show_id'
            return next_show_id, other_next_show_id

In [0]:
def get_table_name(client_name, vendor, start_time):
    table_name = f'existing_all_comm_{client_name}_{vendor.lower()}_'
    table_name += start_time.replace('-', '_').replace(':', ' ').split(' ')[0]
    table_name += f"_{start_time.replace('-', '_').replace(':', ' ').split(' ')[1]}"
    return table_name

In [0]:
start_time = '2025-06-17 15:00:00'
end_time = '2025-06-17 16:00:00'

client_name = 'ispot'
comm_client = 'ispot'
vendor = 'TIVO'
other_vendor = 'TMS' if vendor == 'TIVO' else 'TIVO'

schema_name = 'dev.mohit_gangwani'
table_name = get_table_name(client_name, vendor, start_time)

prev_station_id, other_prev_station_id = prev_next_cols(vendor, 'prev', 'station')
next_station_id, other_next_station_id = prev_next_cols(vendor, 'next', 'station')
prev_show_id, other_prev_show_id = prev_next_cols(vendor, 'prev', 'show')
next_show_id, other_next_show_id = prev_next_cols(vendor, 'next', 'show')

In [0]:
print(f'start_time = {start_time}\nend_time = {end_time}\nclient_name = {client_name}\ncomm_client = {comm_client}')
print(f'vendor = {vendor}\nother_vendor = {other_vendor}\nschema_name = {schema_name}\ntable_name = {table_name}')

In [0]:
drop_sql = f"DROP TABLE IF EXISTS {schema_name}.{table_name};"
create_sql = f"CREATE TABLE IF NOT EXISTS {schema_name}.{table_name} ({get_column_names(vendor)});"

In [0]:
spark.sql(drop_sql.format(schema_name=schema_name, table_name=table_name))

In [0]:
spark.sql(create_sql.format(schema_name=schema_name, table_name=table_name, vendor=vendor))

In [0]:
spark.sql(f"""
INSERT INTO {schema_name}.{table_name} (
    {column_names_insert(vendor)}
)
WITH commercial_id_external_firehose AS (
  SELECT m.fk_commercial_id, m.external_id, m.brand_name, m.title, m.duration
  FROM prod.detection.commercial_id_external_firehose AS m
  JOIN prod.detection.clients cl
    ON m.fk_client_id = cl.client_id
  WHERE CASE WHEN 'nielsen' = '{client_name}' THEN client_name IN ('{comm_client}', 'nielsen') ELSE client_name = '{comm_client}' END
  GROUP BY ALL
)
, nielsen_replacement_national_nyc_alias AS (
    SELECT rl.station_id, rl.fk_show_id, rl.tuner_channel_id, rl.tuner_program_id
    FROM prod.detection.nielsen_replacement_national_nyc AS rl
    JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
      ON bl.station_id = rl.station_id
     AND bl.blacklist_end >= '{start_time}'
    GROUP BY 1,2,3,4
)
, nielsen_replacement_local_alias AS (
    SELECT rl.station_id, rl.fk_show_id, rl.dma_id, rl.tuner_channel_id, rl.tuner_program_id
    FROM prod.detection.nielsen_replacement_local AS rl
    JOIN prod.detection.nielsen_only_distribution_blacklist AS bl
      ON bl.station_id = rl.station_id
     AND bl.blacklist_end >= '{start_time}'
    GROUP BY 1,2,3,4,5
)
, activity_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_activity_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_activity_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = '{client_name}'
    WHERE override.app_name IS NULL
)
, viewing_obfuscation AS (
    SELECT blocked_apps.app_name
    FROM prod.detection.app_viewing_distribution_blacklist AS blocked_apps
    LEFT JOIN prod.detection.app_customer_viewing_distribution_override override
        ON blocked_apps.app_name = override.app_name
        AND override.client_name = '{client_name}'
    WHERE override.app_name IS NULL
),
epg_program_aggregate AS (
    SELECT DISTINCT *
    FROM prod.detection.vizio_epg_program_aggregate 
),
viewing_content_firehose AS (
  SELECT DISTINCT fk_tvid, session_start, session_end, fk_content_id, is_live
  FROM prod.detection.viewing_content_firehose AS content
  WHERE content.session_start >= '{start_time}'::timestamp
      AND content.session_start < '{end_time}'::timestamp
      AND content.partition_key >= '{start_time}'::timestamp::DATE
      AND content.partition_key <= '{end_time}'::timestamp::DATE
)
, content_ids_firehose AS (
  SELECT * FROM detection.content_ids_firehose AS cid
  WHERE content_id IN (
      SELECT DISTINCT c.fk_content_id
      FROM viewing_content_firehose AS c
  )
), station_distribution_blacklist AS (
    WITH agg AS (
        SELECT vendor_station_id, vendor_name, CONCAT_WS(',', SORT_ARRAY(COLLECT_LIST(client_name), false)) AS cl_list
        FROM prod.detection.station_distribution_obfuscation_overwrite
        GROUP BY 1, 2)
    SELECT vendor_station_id AS station_id, vendor_name
    FROM agg
    WHERE cl_list NOT ILIKE '%{client_name}%'
),
inscape_map_deduped AS (
    SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
    FROM (
        SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, ROW_NUMBER() OVER (PARTITION BY mapped_vendor, mapped_vendor_station_id ORDER BY created_at DESC) AS rn
        FROM prod.detection.inscape_station_map) ism
    WHERE ism.rn = 1
)
SELECT 
/*+ BROADCAST(m), 
  BROADCAST(cl), 
  BROADCAST(tp), 
  BROADCAST(pop),
  BROADCAST(location), 
  BROADCAST(epg_program_aggregate), 
  BROADCAST(prev_cid), 
  BROADCAST(next_cid), 
  BROADCAST(next_map),
  BROADCAST(next_vizio_station),
  BROADCAST(next_vizio_program),
  BROADCAST(prev_vizio_station),
  BROADCAST(prev_vizio_program),
  BROADCAST(next_program),
  BROADCAST(next_station),
  BROADCAST(next_program),
  BROADCAST(next_program_alt),
  BROADCAST(prev_program_alt),
  BROADCAST(prev_map),
  BROADCAST(prev_station),
  BROADCAST(prev_program) */
DISTINCT 
    COALESCE(tv.long_tvid, tv.vizio_tvid) AS TVID, 
    '', 
    NULLIF(location.zipcode, ''), 
    REPLACE(dma.dma_name, ',', ''), 
    m.external_id, 
    c.media_time_start, 
    c.session_start, 
    c.session_end, 
    NULLIF(CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN prev_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_prev.station_id, rep_nyc_nat_prev.station_id) IS NULL OR prev_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN prev_station_blacklist.station_id IS NOT NULL THEN NULL 
        WHEN COALESCE({prev_station_id}, {other_prev_station_id}) = 0 THEN coalesce(prev_filecontent.external_id,SPLIT(prev_cid.content_cid, '_')[0]) 
        WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station = '98989898989898' THEN NULL
        WHEN prev_program.database_key IS NOT NULL THEN prev_program.database_key 
        WHEN '{vendor}' = 'TMS' AND prev_vizio_program.program_tms_id IS NOT NULL THEN prev_vizio_program.program_tms_id
        ELSE NULL
    END,''), 
    CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN prev_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_prev.station_id, rep_nyc_nat_prev.station_id) IS NULL OR prev_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN prev_station_blacklist.station_id IS NOT NULL THEN NULL
        WHEN COALESCE({prev_station_id}, {other_prev_station_id}) = 0 THEN prev_filecontent.title
        WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station = '98989898989898' THEN NULL
        ELSE REPLACE(COALESCE(prev_program.title, prev_program_backup.title,
        CASE WHEN prev_vizio_program.series_aggregate_title IS NOT NULL AND prev_vizio_program.series_aggregate_title != '' THEN prev_vizio_program.series_aggregate_title
        ELSE prev_vizio_program.title
        END), ',', '')
    END, 
    c.prev_session_start, 
    c.prev_session_end, 
    CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN prev_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_prev.station_id, rep_nyc_nat_prev.station_id) IS NULL OR prev_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN prev_station_obfs.station_id IS NOT NULL THEN NULL
        WHEN COALESCE({prev_station_id}, {other_prev_station_id}) IS NOT NULL THEN COALESCE(prev_map.inscape_call_sign, prev_map_backup.inscape_call_sign)
        WHEN (prev_station_id = 0) THEN COALESCE(SPLIT(prev_cid.content_cid, '_')[1],'FILE') 
        ELSE NULL
    END, 
    CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN prev_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_prev.station_id, rep_nyc_nat_prev.station_id) IS NULL OR prev_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN prev_station_obfs.station_id IS NOT NULL THEN NULL
        WHEN {prev_station_id} IS NOT NULL THEN
             CASE WHEN (prev_station.inscape_station_name IS NOT NULL) THEN prev_station.inscape_station_name
                  WHEN (LOWER(prev_station.station_affil) LIKE '%affiliate%'
                        OR LOWER(prev_station.station_affil) LIKE '%independent%'
                        OR LOWER(prev_station.station_affil) LIKE '%low power%')
                       THEN prev_station.station_affil END
        WHEN {prev_station_id} IS NULL AND {other_prev_station_id} IS NOT NULL THEN
             CASE WHEN (prev_station_backup.inscape_station_name IS NOT NULL) THEN prev_station_backup.inscape_station_name
                  WHEN (LOWER(prev_station_backup.station_affil) LIKE '%affiliate%'
                        OR LOWER(prev_station_backup.station_affil) LIKE '%independent%'
                        OR LOWER(prev_station_backup.station_affil) LIKE '%low power%')
                       THEN prev_station_backup.station_affil END
        WHEN prev_chanb.channel_name IS NOT NULL OR c.prev_vizio_epg_station = '98989898989898' THEN 'OBFUSCATED'
        WHEN c.prev_vizio_epg_station IS NOT NULL THEN prev_vizio_station.name
        ELSE NULL
    END, 
    NULLIF(CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN next_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_next.station_id, rep_nyc_nat_next.station_id) IS NULL OR next_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN next_station_blacklist.station_id IS NOT NULL THEN NULL
        WHEN COALESCE({next_station_id}, {other_next_station_id}) = 0 THEN coalesce(next_filecontent.external_id,SPLIT(next_cid.content_cid, '_')[0]) 
        WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station = '98989898989898' THEN NULL
        WHEN next_program.database_key IS NOT NULL THEN next_program.database_key
        WHEN '{vendor}' = 'TMS' AND next_vizio_program.program_tms_id IS NOT NULL THEN next_vizio_program.program_tms_id
        ELSE NULL
    END,''), 
    CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN next_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_next.station_id, rep_nyc_nat_next.station_id) IS NULL OR next_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN next_station_blacklist.station_id IS NOT NULL THEN NULL
        WHEN COALESCE({next_station_id}, {other_next_station_id}) = 0 THEN next_filecontent.title
        WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station = '98989898989898' THEN NULL
        ELSE REPLACE(COALESCE(next_program.title, next_program_backup.title,
        CASE WHEN next_vizio_program.series_aggregate_title IS NOT NULL AND next_vizio_program.series_aggregate_title != '' THEN next_vizio_program.series_aggregate_title
        ELSE next_vizio_program.title
        END), ',', '')
    END, 
    c.next_session_start, 
    c.next_session_end, 
    CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN next_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_next.station_id, rep_nyc_nat_next.station_id) IS NULL OR next_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN next_station_obfs.station_id IS NOT NULL THEN NULL
        WHEN COALESCE({next_station_id}, {other_next_station_id}) IS NOT NULL THEN COALESCE(next_map.inscape_call_sign, next_map_backup.inscape_call_sign)
        WHEN COALESCE({next_station_id}, {other_next_station_id}) = 0 THEN COALESCE(SPLIT(next_cid.content_cid, '_')[1],'FILE')
    END, 
    CASE
    WHEN (cl2.client_id is not NULL) THEN NULL
    WHEN next_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_next.station_id, rep_nyc_nat_next.station_id) IS NULL OR next_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
    WHEN next_station_obfs.station_id IS NOT NULL THEN NULL
    WHEN {next_station_id} IS NOT NULL THEN
         CASE WHEN (next_station.inscape_station_name IS NOT NULL) THEN next_station.inscape_station_name
              WHEN (LOWER(next_station.station_affil) LIKE '%affiliate%'
                    OR LOWER(next_station.station_affil) LIKE '%independent%'
                    OR LOWER(next_station.station_affil) LIKE '%low power%')
                   THEN next_station.station_affil END
    WHEN {next_station_id} IS NULL AND {other_next_station_id} IS NOT NULL THEN
         CASE WHEN (next_station_backup.inscape_station_name IS NOT NULL) THEN next_station_backup.inscape_station_name
              WHEN (LOWER(next_station_backup.station_affil) LIKE '%affiliate%'
                    OR LOWER(next_station_backup.station_affil) LIKE '%independent%'
                    OR LOWER(next_station_backup.station_affil) LIKE '%low power%')
                   THEN next_station_backup.station_affil END
    WHEN next_chanb.channel_name IS NOT NULL OR c.next_vizio_epg_station = '98989898989898' THEN 'OBFUSCATED'
    WHEN c.next_vizio_epg_station IS NOT NULL THEN next_vizio_station.name
    ELSE NULL
    END, 
    CASE
        WHEN (cl2.client_id is not NULL) THEN NULL
        WHEN tvis.category = 'APPS' and tis.app_name = 'OBFUSCATED' AND c.prev_vizio_epg_station IS NOT NULL THEN 't'
        WHEN prev_nielsen_blacklist.station_id IS NOT NULL AND (NVL(rep_local_prev.station_id, rep_nyc_nat_prev.station_id) IS NULL OR prev_nielsen_blacklist.ingest_time IS NOT NULL) THEN NULL
        WHEN prev_station_blacklist.station_id IS NOT NULL THEN NULL
        ELSE CASE WHEN prev_content.is_live = TRUE THEN 't' WHEN prev_content.is_live = FALSE THEN 'f' ELSE NULL END
    END, 
    REPLACE(m.brand_name, ',', ''), 
    REPLACE(m.title, ',', ''), 
    m.duration, 
    ip.ip_address, 
    tvis.category, 
    tvis.input_device, 
    CASE WHEN UPPER(tvis.category) = 'APPS' THEN 
            CASE WHEN c.prev_vizio_epg_station IS NOT NULL THEN 'WatchFree+'
                WHEN c.prev_vizio_epg_station IS NULL AND tis.app_name = 'WatchFree+' THEN 'OBFUSCATED'
                WHEN appb.app_name IS NOT NULL THEN 'OBFUSCATED'
                WHEN LOWER(coalesce(tis.app_name)) IN ('cbs all access', 'cbs news', 'paramount+', 'fandangonow', 'nbc', 'tnt', 'watch tbs', 'iheartradio','pandora','tv games') AND c.prev_show_id IS NOT NULL THEN NULL
                WHEN LOWER(coalesce(tis.app_name)) = 'unknown' THEN NULL
                ELSE coalesce(tis.app_name) END
            WHEN prev_content.is_live = true AND LOWER(tvis.input_device) in ('xbox','amazon_fire','apple_tv','chromecast','nintendo switch','playstation 4','playstation 5','roku') THEN 'vMVPD'
        END
    FROM prod.detection.viewing_commercials_firehose AS c
    JOIN prod.detection.zoo AS z 
        ON c.fk_zoo_id = z.zoo_id
        AND z.zoo = 'control-zoo-dtsprod.tvinteractive.tv'
    INNER JOIN
        prod.detection.tv AS tv
        ON c.fk_tvid = tv.tvid
        AND tv.oem = 'VIZIO'
    JOIN
        prod.detection.tv_populations AS tp
        ON c.fk_tvid = tp.fk_tvid 
    JOIN
        prod.detection.populations AS pop
        ON tp.fk_population_id = pop.population_id 
        AND LOWER(pop.population_name) = 'opted_in'
    JOIN
        commercial_id_external_firehose AS m
        ON c.fk_commercial_id = m.fk_commercial_id
    JOIN
        prod.detection.location AS location
        ON c.fk_location_id = location.location_id
        AND UPPER(location.country_code) = 'US'
    JOIN 
        prod.detection.tv_input_stats_firehose  tvis    
        ON c.session_start >= tvis.create_timestamp 
        AND c.session_start < tvis.next_create_timestamp
        AND tvis.create_timestamp <= '{end_time}'::timestamp
        AND tvis.next_create_timestamp >= '{start_time}'::timestamp
        AND c.fk_tvid = tvis.fk_tvid   
        AND c.fk_input_source_id = tvis.fk_input_source_id
    JOIN
        prod.detection.tv_settings AS tv_settings
        ON c.session_start < tv_settings.next_create_timestamp
        AND c.session_start >= tv_settings.create_timestamp
        AND c.fk_tvid = tv_settings.fk_tvid
        AND tv_settings.create_timestamp <= '{end_time}'::timestamp
        AND tv_settings.next_create_timestamp >= '{start_time}'::timestamp
    JOIN 
        prod.detection.settings AS settings
        ON tv_settings.fk_settings_id = settings.settings_id
        AND UPPER(settings.country_name) = 'USA'
    LEFT OUTER JOIN
        prod.detection.dma AS dma ON c.fk_dma_id = dma.dma_id
    LEFT OUTER JOIN prod.detection.vizio_epg_station AS prev_vizio_station
        ON TRY_CAST(c.prev_vizio_epg_station AS STRING) <=> TRY_CAST(prev_vizio_station.station_id AS STRING)
    LEFT OUTER JOIN epg_program_aggregate AS prev_vizio_program
        ON TRY_CAST(c.prev_vizio_epg_program AS STRING) <=> TRY_CAST(prev_vizio_program.program_aggregate_id AS STRING)
        AND TRY_CAST(c.prev_vizio_epg_program AS STRING) NOT IN ('0', '', '-1')
        AND c.prev_vizio_epg_program IS NOT NULL
    LEFT OUTER JOIN prod.detection.vizio_epg_station AS next_vizio_station
        ON TRY_CAST(c.next_vizio_epg_station AS STRING) <=> TRY_CAST(next_vizio_station.station_id AS STRING)
    LEFT OUTER JOIN epg_program_aggregate AS next_vizio_program
        ON TRY_CAST(c.next_vizio_epg_program AS STRING) <=> TRY_CAST(next_vizio_program.program_aggregate_id AS STRING)
        AND TRY_CAST(c.next_vizio_epg_program AS STRING) NOT IN ('0', '', '-1')
        AND c.next_vizio_epg_program IS NOT NULL
    LEFT OUTER JOIN viewing_content_firehose AS prev_content 
        ON c.fk_tvid = prev_content.fk_tvid
        AND prev_content.session_start = c.prev_session_start
    LEFT OUTER JOIN prod.detection.epg_station AS prev_station  
        ON prev_station.station_id = {prev_station_id}
        AND prev_station.vendor_name = '{vendor}'
    LEFT OUTER JOIN prod.detection.epg_station AS prev_station_backup
        ON prev_station_backup.station_id = {other_prev_station_id}
        AND {prev_station_id} IS NULL
        AND prev_station_backup.vendor_name = '{other_vendor}'
    LEFT OUTER JOIN inscape_map_deduped AS prev_map
        ON prev_map.mapped_vendor_station_id = {prev_station_id}
        AND prev_map.mapped_vendor = '{vendor}'
    LEFT OUTER JOIN inscape_map_deduped AS prev_map_backup
        ON prev_map_backup.mapped_vendor_station_id = {other_prev_station_id}
        AND {prev_station_id} IS NULL
        AND prev_map_backup.mapped_vendor = '{other_vendor}' 
    LEFT OUTER JOIN station_distribution_blacklist AS prev_station_blacklist
        ON {prev_station_id} = prev_station_blacklist.station_id
        AND prev_station_blacklist.vendor_name = prev_map.mapped_vendor
    LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS prev_station_obfs
        ON {prev_station_id} = prev_station_obfs.vendor_station_id
        AND prev_station_obfs.vendor_name = prev_map.mapped_vendor
    LEFT OUTER JOIN prod.detection.epg_show AS prev_program
        ON {prev_show_id} = prev_program.show_id
       AND prev_program.vendor_name = '{vendor}'
    LEFT OUTER JOIN prod.detection.epg_show AS prev_program_backup
        ON {other_prev_show_id} = prev_program_backup.show_id
        AND prev_program_backup.vendor_name = '{other_vendor}'
        AND {prev_show_id} IS NULL
    LEFT OUTER JOIN prod.detection.content_id_external_firehose AS prev_filecontent 
        ON {prev_show_id} = prev_filecontent.fk_content_id
    LEFT OUTER JOIN prod.detection.content_id_external_firehose AS next_filecontent 
        ON {next_show_id} = next_filecontent.fk_content_id
    LEFT OUTER JOIN prod.detection.content_id_external_firehose AS m_filter 
        ON m_filter.fk_content_id = prev_content.fk_content_id
    LEFT OUTER JOIN content_ids_firehose as prev_cid on c.prev_show_id = prev_cid.content_id
    LEFT OUTER JOIN content_ids_firehose as next_cid on c.next_show_id = next_cid.content_id
    LEFT OUTER JOIN prod.detection.clients cl2
        ON m_filter.fk_client_id = cl2.client_id
        AND cl2.client_name <> '{comm_client}'
    LEFT OUTER JOIN prod.detection.epg_station AS next_station  
        ON next_station.station_id = {next_station_id}
        AND next_station.vendor_name = '{vendor}'
    LEFT OUTER JOIN prod.detection.epg_station AS next_station_backup
        ON next_station_backup.station_id = {other_next_station_id}
        AND {next_station_id} IS NULL
        AND next_station_backup.vendor_name = '{other_vendor}'
    LEFT OUTER JOIN inscape_map_deduped AS next_map
        ON next_map.mapped_vendor_station_id = {next_station_id}
        AND next_map.mapped_vendor = '{vendor}'
    LEFT OUTER JOIN inscape_map_deduped AS next_map_backup
        ON next_map_backup.mapped_vendor_station_id = {other_next_station_id}
        AND {next_station_id} IS NULL
        AND next_map_backup.mapped_vendor = '{other_vendor}' 
    LEFT OUTER JOIN station_distribution_blacklist AS next_station_blacklist
        ON {next_station_id} = next_station_blacklist.station_id
        AND next_station_blacklist.vendor_name = next_map.mapped_vendor
    LEFT OUTER JOIN prod.detection.station_metadata_obfuscation AS next_station_obfs
        ON {next_station_id} = next_station_obfs.vendor_station_id
        AND next_station_obfs.vendor_name = next_map.mapped_vendor
    LEFT OUTER JOIN prod.detection.epg_show AS next_program
        ON {next_show_id} = next_program.show_id
       AND next_program.vendor_name = '{vendor}'
    LEFT OUTER JOIN prod.detection.epg_show AS next_program_backup
        ON {other_next_show_id} = next_program_backup.show_id
       AND next_program_backup.vendor_name = '{other_vendor}'
        AND {next_show_id} IS NULL
    LEFT OUTER JOIN prod.detection.tv_inputsource tis    
        ON  c.session_start >=  (tis.create_timestamp::double)::timestamp
        AND c.session_start <  (tis.next_create_timestamp::double)::timestamp
        AND tis.create_timestamp <= ('{end_time}'::timestamp::double)::timestamp
        AND tis.next_create_timestamp >= ('{start_time}'::timestamp::double)::timestamp
        AND c.fk_tvid = tis.fk_tvid 
        AND c.fk_input_source_id = tis.fk_input_source_id
    LEFT OUTER JOIN activity_obfuscation appb 
        ON coalesce(tis.app_name) = appb.app_name 
    LEFT OUTER JOIN viewing_obfuscation AS acrb
        ON coalesce(tis.app_name) = acrb.app_name
    LEFT OUTER JOIN 
        prod.detection.free_channels_distribution_blacklist prev_chanb 
        ON prev_vizio_station.name = prev_chanb.channel_name 
    LEFT OUTER JOIN 
        prod.detection.free_channels_distribution_blacklist next_chanb 
        ON next_vizio_station.name = next_chanb.channel_name
    LEFT OUTER JOIN
        prod.detection.tv_ip_address AS ip
        ON c.session_start >= ip.create_timestamp
        AND c.session_start < ip.next_create_timestamp
        AND ip.create_timestamp <= '{end_time}'::timestamp
        AND ip.next_create_timestamp >= '{start_time}'::timestamp
        AND c.fk_tvid = ip.fk_tvid
    LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS prev_nielsen_blacklist
        ON prev_nielsen_blacklist.station_id = prev_map.inscape_station_id 
        AND c.session_start >= prev_nielsen_blacklist.blacklist_start 
        AND c.session_start < prev_nielsen_blacklist.blacklist_end 
        AND '{client_name}' != 'nielsen'
    LEFT OUTER JOIN prod.detection.nielsen_only_distribution_blacklist AS next_nielsen_blacklist
        ON next_nielsen_blacklist.station_id = next_map.inscape_station_id 
        AND c.session_start >= next_nielsen_blacklist.blacklist_start 
        AND c.session_start < next_nielsen_blacklist.blacklist_end
        AND '{client_name}' != 'nielsen'
    LEFT OUTER JOIN nielsen_replacement_local_alias AS rep_local_prev
        ON prev_map.inscape_station_id  = rep_local_prev.station_id
        AND {prev_show_id} = rep_local_prev.fk_show_id
        AND c.fk_dma_id = rep_local_prev.dma_id
        AND '{client_name}' != 'nielsen'
    LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS rep_nyc_nat_prev
        ON prev_map.inscape_station_id  = rep_nyc_nat_prev.station_id
        AND {prev_show_id} = rep_nyc_nat_prev.fk_show_id
        AND '{client_name}' != 'nielsen'
    LEFT OUTER JOIN nielsen_replacement_local_alias AS rep_local_next
        ON next_map.inscape_station_id  = rep_local_next.station_id
        AND {next_show_id} = rep_local_next.fk_show_id
        AND c.fk_dma_id = rep_local_next.dma_id
        AND '{client_name}' != 'nielsen'
    LEFT OUTER JOIN nielsen_replacement_national_nyc_alias AS rep_nyc_nat_next
        ON next_map.inscape_station_id  = rep_nyc_nat_next.station_id
        AND {next_show_id} = rep_nyc_nat_next.fk_show_id
        AND '{client_name}' != 'nielsen'
    WHERE
        c.session_start >= '{start_time}'::timestamp
        AND c.session_start < '{end_time}'::timestamp
        AND c.partition_key >= '{start_time}'::timestamp::DATE
        AND c.partition_key <= '{end_time}'::timestamp::DATE
        AND CASE WHEN tvis.category = 'APPS' AND prev_vizio_station.name IS NULL AND acrb.app_name IS NOT NULL THEN FALSE ELSE TRUE END;""")